# Lecture 5 — `struct`

Today’s goal:

> Group related values together into one object.

This is the first big bridge toward real Autoware metric code.

## 1. Why `struct`?

A metric result usually needs several values:

```text
score
available
reason
```

If we keep them as separate variables, they can become inconsistent:

```cpp
double score = 0.0;
bool available = true;
std::string reason = "available";
```

That is confusing: score says fail, reason says pass.

A `struct` lets us package these values together.

## 2. Basic `struct` syntax

```cpp
struct MetricResult
{
  double score;
  bool available;
  std::string reason;
};
```

Think of it like a small custom data type.

In [ ]:
#include <iostream>
#include <string>

struct MetricResult05A
{
  double score;
  bool available;
  std::string reason;
};

{
  MetricResult05A result;
  result.score = 1.0;
  result.available = true;
  result.reason = "available";

  std::cout << std::boolalpha;
  std::cout << "score = " << result.score << std::endl;
  std::cout << "available = " << result.available << std::endl;
  std::cout << "reason = " << result.reason << std::endl;
}

## 3. Dot access: `result.score`

When you have a struct object:

```cpp
MetricResult result;
```

You access fields using dot:

```cpp
result.score
result.available
result.reason
```

## 4. Default member values

In metric code, default values are very useful.

```cpp
struct MetricResult
{
  double score{0.0};
  bool available{false};
  std::string reason{"unavailable"};
};
```

Now a new result starts as unavailable by default.

In [ ]:
#include <iostream>
#include <string>

struct MetricResult05B
{
  double score{0.0};
  bool available{false};
  std::string reason{"unavailable"};
};

{
  MetricResult05B result;

  std::cout << std::boolalpha;
  std::cout << "default score = " << result.score << std::endl;
  std::cout << "default available = " << result.available << std::endl;
  std::cout << "default reason = " << result.reason << std::endl;
}

This default pattern is excellent for metrics:

```text
Start unavailable.
If checks pass, set available = true.
If violation happens, set score = 0.
```

## 5. Function returning a struct

Now one function can return `score`, `available`, and `reason` together.

In [ ]:
#include <iostream>
#include <string>

struct MetricResult05C
{
  double score{0.0};
  bool available{false};
  std::string reason{"unavailable"};
};

MetricResult05C calculate_speed_metric_05C(double ego_speed_mps, double speed_limit_mps)
{
  MetricResult05C result;

  result.available = true;
  result.score = 1.0;
  result.reason = "available";

  if (ego_speed_mps > speed_limit_mps) {
    result.score = 0.0;
    result.reason = "speed_too_high";
    return result;
  }

  return result;
}

{
  const auto result = calculate_speed_metric_05C(12.0, 10.0);

  std::cout << std::boolalpha;
  std::cout << "score = " << result.score << std::endl;
  std::cout << "available = " << result.available << std::endl;
  std::cout << "reason = " << result.reason << std::endl;
}

## 6. Early return with struct

This is the real metric shape:

```cpp
MetricResult result;

if (bad_input) {
  result.reason = "unavailable_bad_input";
  return result;
}

result.available = true;
result.score = 1.0;
result.reason = "available";

if (violation) {
  result.score = 0.0;
  result.reason = "violation";
  return result;
}

return result;
```

In [ ]:
#include <iostream>
#include <string>

struct MetricResult05D
{
  double score{0.0};
  bool available{false};
  std::string reason{"unavailable"};
};

MetricResult05D calculate_ttc_metric_05D(bool has_ttc_value, double ttc_s, double threshold_s)
{
  MetricResult05D result;

  if (!has_ttc_value) {
    result.reason = "unavailable_no_ttc_value";
    return result;
  }

  result.available = true;
  result.score = 1.0;
  result.reason = "available";

  if (ttc_s < threshold_s) {
    result.score = 0.0;
    result.reason = "ttc_too_low";
    return result;
  }

  return result;
}

{
  const auto result = calculate_ttc_metric_05D(true, 1.5, 2.0);

  std::cout << std::boolalpha;
  std::cout << "score = " << result.score << std::endl;
  std::cout << "available = " << result.available << std::endl;
  std::cout << "reason = " << result.reason << std::endl;
}

## 7. Autoware connection

Autoware metrics often have result structs like:

```cpp
struct SomeMetricResult
{
  double score{0.0};
  bool available{false};
  std::string reason{"unavailable"};
  double infraction_time_s{...};
};
```

So this lecture is directly preparing you to read real metric code.

## 8. Homework

Create a `LaneKeepingResult05` struct and a function:

```text
calculate_lane_keeping_metric(has_lane_info, lateral_deviation_m, max_lateral_deviation_m)
```

Rules:

```text
If no lane info:
  available = false
  reason = unavailable_no_lane_info

If lateral deviation too large:
  available = true
  score = 0
  reason = lane_deviation_too_large

Otherwise:
  available = true
  score = 1
  reason = available
```

In [ ]:
#include <iostream>
#include <string>

struct LaneKeepingResult05
{
  double score{0.0};
  bool available{false};
  std::string reason{"unavailable"};
};

LaneKeepingResult05 calculate_lane_keeping_metric_05(
  bool has_lane_info, double lateral_deviation_m, double max_lateral_deviation_m)
{
  LaneKeepingResult05 result;

  // TODO: implement logic here.

  return result;
}

{
  const auto result = calculate_lane_keeping_metric_05(true, 0.7, 0.5);
  std::cout << std::boolalpha;
  std::cout << "score = " << result.score << std::endl;
  std::cout << "available = " << result.available << std::endl;
  std::cout << "reason = " << result.reason << std::endl;
}

Next lecture: **`std::vector`** — list-like containers, which prepare us for `trajectory.points`.